In [6]:
import pandas as pd
from collections import Counter
from sklearn.feature_extraction.text import CountVectorizer

# Load the data
df = pd.read_csv("../data/Reclamations_clustered.csv")

# Ensure 'cluster' is treated as string for grouping
df['cluster'] = df['cluster'].astype(str)

# Function to get most common values per column
def most_common(series, n=3):
    return series.value_counts().head(n).to_dict()

# Function to extract top keywords in texte_reclamation
FRENCH_STOP_WORDS = [
    'le', 'la', 'les', 'de', 'des', 'et', 'un', 'une', 'du', 
    'dans', 'en', 'pour', 'que', 'qui', 'à', 'au', 'aux', 'ce', 
    'ces', 'il', 'elle', 'on', 'nous', 'vous', 'ils', 'elles'
]

def top_keywords(text_series, n=5):
    text_series = text_series.dropna().astype(str)
    if len(text_series) == 0:
        return {}
    
    vectorizer = CountVectorizer(stop_words=FRENCH_STOP_WORDS)
    X = vectorizer.fit_transform(text_series)
    counts = X.toarray().sum(axis=0)
    keywords = pd.Series(counts, index=vectorizer.get_feature_names_out())
    
    return keywords.sort_values(ascending=False).head(n).to_dict()

# Profile each cluster
cluster_profiles = []

for cluster, group in df.groupby('cluster'):
    profile = {
        'cluster': cluster,
        'size': len(group),
        'top_categories': most_common(group['category']),
        'top_sentiments': most_common(group['sentiment']),
        'top_urgency': most_common(group['urgency']),
        'top_status': most_common(group['status']),
        'top_suggested_actions': most_common(group['suggested_action']),
        'top_keywords': top_keywords(group['texte_reclamation'])
    }
    cluster_profiles.append(profile)

# Pretty-print each cluster
for profile in cluster_profiles:
    print("="*60)
    print(f"Cluster: {profile['cluster']}  |  Size: {profile['size']}")
    print("-"*60)
    print("Top Categories:", ", ".join(f"{k} ({v})" for k, v in profile['top_categories'].items()))
    print("Top Sentiments:", ", ".join(f"{k} ({v})" for k, v in profile['top_sentiments'].items()))
    print("Top Urgency:", ", ".join(f"{k} ({v})" for k, v in profile['top_urgency'].items()))
    print("Top Status:", ", ".join(f"{k} ({v})" for k, v in profile['top_status'].items()))
    print("Top Suggested Actions:", ", ".join(f"{k} ({v})" for k, v in profile['top_suggested_actions'].items()))
    print("Top Keywords:", ", ".join(f"{k} ({v})" for k, v in profile['top_keywords'].items()))
    print("="*60 + "\n")


Cluster: 0  |  Size: 1986
------------------------------------------------------------
Top Categories: Coupure De Ligne (445), Internet Lent (407), Facturation (211)
Top Sentiments: Négatif (1193), Neutre (590), Positif (203)
Top Urgency: Moyenne (803), Basse (794), Élevée (307)
Top Status: Résolu (720), En cours (606), Nouveau (464)
Top Suggested Actions: Envoi technicien (490), Redémarrage du modem (370), Vérification facturation (310)
Top Keywords: client (1986), continue (1986), facturation (1986), lors (1986), problème (1986)

Cluster: 1  |  Size: 2002
------------------------------------------------------------
Top Categories: Coupure De Ligne (444), Internet Lent (410), Facturation (239)
Top Sentiments: Négatif (1213), Neutre (594), Positif (195)
Top Urgency: Basse (804), Moyenne (786), Élevée (320)
Top Status: Résolu (726), En cours (580), Nouveau (515)
Top Suggested Actions: Envoi technicien (501), Redémarrage du modem (407), Vérification facturation (284)
Top Keywords: abonné